In [1]:
import pdfplumber
import pandas as pd
import os
import re
from img2table.document import Image
from io import BytesIO
from pdf2image import convert_from_path
from img2table.ocr import TesseractOCR
from img2table.document import PDF
import json

In [2]:
ocr = TesseractOCR()

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.58 : libtiff 4.7.2 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.8 zlib/1.2.12 liblzma/5.8.3 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.3 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.64.0


In [3]:
def extract_headers(page):
    lines = page.extract_text_lines()
    return pd.DataFrame([line for line in lines if re.search('^[A-Z]{1}[0-9]{1,2}', line['text']) is not None])

In [4]:
def select_best_heading(page, table, headers):
    if len(headers) == 0:
        return ''
    headers['is_above'] = headers['top'].apply(lambda x: x < table.bbox.relative.y1 * page.height)
    headers = headers[headers['is_above']].sort_values('top', ascending=False).reset_index(drop=True)
    if len(headers) == 0:
        return ''
    return headers['text'][0]

In [5]:
def parse_page_tables(file, page_index):
    pdf = pdfplumber.open(file)
    page = pdf.pages[page_index]
    headers = extract_headers(page)

    doc = PDF(file,
          pages=[page_index],
          detect_rotation=False,
          pdf_text_extraction=True)

    results = doc.extract_tables(ocr=ocr,
                    implicit_rows=False,
                    implicit_columns=False,
                    borderless_tables=False,
                    min_confidence=50,
                    max_workers=1)

    data = []
    for table in results[page_index]:
        heading = select_best_heading(page, table, headers)
        code_match = re.search('^[A-Z]{1}[0-9]+', heading)
        if code_match is not None:
            code = code_match.group(0).strip()
        else:
            code = ''

        heading = heading.replace(code, '')
        heading = re.sub('^\\.', '', heading)
        heading = heading.strip()

        data.append({
            'table_code': code,
            'table_title': heading,
            'table_records': table.df.fillna('').to_dict(orient='records'),
            'table_html': re.sub('[\n ]+', ' ', table.html)
        })

    return data

In [6]:
def extract_data_from_file(file):
    year = file.split('/')[-2].strip()
    print(file)
    try:
        pdf = pdfplumber.open(file)
    except:
        return None
    unitid = file.split('/')[-1].replace('.pdf', '').strip()

    data = []
    for page in range(0, len(pdf.pages)):
        try:
            data.extend(parse_page_tables(file, page))
        except:
            continue

    if len(data) == 0:
        return None

    data = pd.DataFrame(data)

    data = data[data['table_code'] != '']

    data = data.groupby('table_code').agg({
        'table_code': first,
        'table_title': first,
        'table_records': list,
        'table_html': lambda tables: '<div class="separator"></div>'.join(tables)
    }).reset_index(drop=True)

    data['table_num'] = data['table_code'].apply(lambda x: re.search('[0-9]+', x).group(0)).apply(int)
    data['section'] = data['table_code'].apply(lambda x: re.search('[A-Z]+', x).group(0))
    data = data.sort_values(['section', 'table_num'])

    data = data[['section', 'table_num', 'table_code', 'table_title', 'table_records', 'table_html']]
    data['unitid'] = unitid
    data['file'] = file
    data['year'] = year

    return data

In [7]:
def first(lst):
    return list(lst)[0]

In [8]:
directory = pd.read_csv('../data/ipeds/hd2025.csv')
directory = directory[['UNITID', 'INSTNM', 'STABBR']]

In [9]:
years = pd.DataFrame({
    'year': [year for year in os.listdir('../data/cds-docs') if 'DS_Store' not in year and 'no-date' not in year]
})

In [10]:
years['start_year'] = years['year'].apply(lambda x: x.split('-')[0]).apply(int)
years['end_year'] = years['year'].apply(lambda x: x.split('-')[1]).apply(int)

In [11]:
years = years.query('start_year >= 2021 and end_year < 2026')
years = years.sort_values('start_year', ascending=False)
years = years['year']

In [12]:
all_files = []
for year in years:
    all_files.extend([f'../data/cds-docs/{year}/{file}' for file in os.listdir(f'../data/cds-docs/{year}') if '.DS_Store' not in file])

In [13]:
all_files = pd.DataFrame({
    'path': all_files
})

In [14]:
all_files['unitid'] = all_files['path'].apply(lambda x: x.split('/')[-1].replace('.pdf', '').strip())
all_files['year'] = all_files['path'].apply(lambda x: x.split('/')[-2])

In [15]:
all_files = all_files.groupby('unitid').agg({
    'path': list,
    'year': lambda x: list(x)[0]
}).reset_index()

In [16]:
for i in range(0, len(all_files)):
    unitid = all_files['unitid'][i]
    out_path = f'../data/raw-parse/{unitid}.csv'

    if i % 20 == 0:
        print(f'{i / len(all_files)*100}% complete')

    if os.path.exists(out_path):
        print(f'Already scraped {unitid}')
        continue
    else:
        try:
            df = pd.concat([extract_data_from_file(file) for file in all_files['path'][i]])
        except Exception as e:
            print(f'{unitid}, {e}')
            continue
        df.to_csv(out_path, index=False)

0.0% complete
Already scraped 100663
Already scraped 100706
Already scraped 100724
Already scraped 100751
Already scraped 100830
../data/cds-docs/2024-2025/100858.pdf
100858, All objects passed were None
Already scraped 101480
Already scraped 101709
Already scraped 101879
Already scraped 102049
Already scraped 102094
Already scraped 102368
Already scraped 102614
Already scraped 104151
Already scraped 104179
Already scraped 105330
Already scraped 106397
Already scraped 106458
Already scraped 106467
Already scraped 107080
3.2948929159802307% complete
Already scraped 107558
Already scraped 110404
Already scraped 110413
Already scraped 110422
Already scraped 110510
Already scraped 110529
Already scraped 110538
Already scraped 110547
../data/cds-docs/2024-2025/110565.pdf
110565, All objects passed were None
Already scraped 110583
Already scraped 110592
Already scraped 110608
Already scraped 110617
Already scraped 110635
Already scraped 110644
Already scraped 110653
Already scraped 110662
Al

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

232982, All objects passed were None
Already scraped 233277
Already scraped 233374
Already scraped 233541
Already scraped 233718
Already scraped 233897
Already scraped 233921
Already scraped 234030
Already scraped 234076
Already scraped 234085
Already scraped 234207
Already scraped 234827
Already scraped 235097
92.25700164744646% complete
Already scraped 235167
Already scraped 235316
Already scraped 236133
../data/cds-docs/2024-2025/236230.pdf
236230, All objects passed were None
Already scraped 236328
Already scraped 236452
Already scraped 236896
Already scraped 236939
Already scraped 236948
Already scraped 237011
Already scraped 237057
Already scraped 237330
Already scraped 237525
Already scraped 238032
Already scraped 238333
Already scraped 238476
Already scraped 239017
../data/cds-docs/2024-2025/239105.pdf
../data/cds-docs/2023-2024/239105.pdf
../data/cds-docs/2022-2023/239105.pdf
../data/cds-docs/2021-2022/239105.pdf
../data/cds-docs/2024-2025/239318.pdf
../data/cds-docs/2023-2024

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


../data/cds-docs/2022-2023/243744.pdf
../data/cds-docs/2021-2022/243744.pdf
../data/cds-docs/2022-2023/243780.pdf
../data/cds-docs/2021-2022/243780.pdf
../data/cds-docs/2021-2022/245892.pdf
../data/cds-docs/2023-2024/247834.pdf
../data/cds-docs/2022-2023/247834.pdf
../data/cds-docs/2024-2025/262129.pdf
../data/cds-docs/2023-2024/262129.pdf
../data/cds-docs/2022-2023/262129.pdf
../data/cds-docs/2021-2022/262129.pdf
../data/cds-docs/2022-2023/399911.pdf
../data/cds-docs/2024-2025/433660.pdf
../data/cds-docs/2023-2024/433660.pdf
../data/cds-docs/2022-2023/433660.pdf
../data/cds-docs/2021-2022/433660.pdf
../data/cds-docs/2023-2024/441982.pdf
../data/cds-docs/2022-2023/441982.pdf
../data/cds-docs/2021-2022/441982.pdf
../data/cds-docs/2024-2025/443049.pdf
../data/cds-docs/2021-2022/445188.pdf
../data/cds-docs/2024-2025/447689.pdf
../data/cds-docs/2023-2024/447689.pdf
../data/cds-docs/2022-2023/447689.pdf
../data/cds-docs/2021-2022/447689.pdf
98.84678747940691% complete
../data/cds-docs/2022-

In [54]:
df = pd.concat([pd.read_csv(f'../data/raw-parse/{file}') for file in os.listdir('../data/raw-parse')])

In [55]:
df['year_num'] = df['year'].apply(lambda x: int(x.split('-')[0]))
df = df.sort_values('year', ascending=False).reset_index(drop=True)

In [56]:
df['unitid'] = df['unitid'].apply(str)
directory['UNITID'] = directory['UNITID'].apply(str)

In [57]:
df['unitid'] = df['unitid'].apply(lambda x: x.split('_')[0])

In [58]:
data = {}

In [59]:
for unitid in df['unitid'].unique():
    temp_directory = directory.query(f'UNITID == "{unitid}"')

    college_df = df.query(f'unitid == "{unitid}"').reset_index(drop=True)

    if len(temp_directory) == 0:
        print(f'{unitid} not found')
        continue
    inst_data = temp_directory.to_dict(orient='records')[0]
    inst_data['years'] = {}

    for year in college_df['year'].unique():
        temp = college_df.query(f'year == "{year}"').reset_index(drop=True)
        inst_data['years'][year] = {}
        inst_data['years'][year]['file'] = temp['file'][0]
        inst_data['years'][year]['parsed_tables'] = temp.to_dict(orient='records')
    
    data[unitid] = inst_data

    # with open(f'../web-app/colleges/{unitid}.json', 'w') as out_file:
    #     json.dump(data, out_file)

159351 not found
480569 not found


In [60]:
with open('../web-app/data.json', 'w') as out_file:
    json.dump(data, out_file)